<a href="https://colab.research.google.com/github/siddu-09/groq_model_comparison/blob/main/Copy_of_LLM_Lab_Question_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Lab Overview

* **Laboratory Exercise:** This notebook evaluates multiple Groq models and selects the one that gives a correct response with the lowest token usage.

**Hints:**
- Use the same prompt for every model to keep comparison fair
- Compare both correctness and total token usage
- Use deterministic prompts for reliable evaluation

**What each component does:**
| Component | Description |
|----------|-------------|
| `Objective` | Defines the optimization goal: correctness + minimum tokens |
| `Background` | Explains token cost, quality trade-offs, and model variability |
| `Evaluation flow` | Standardizes how each model is tested and compared |

**Documentation:**
- [Groq Docs](https://console.groq.com/docs/overview) - Platform and API overview
- [Tokenization Concepts](https://platform.openai.com/tokenizer) - Token basics (general concept)

## 2. Setup Environment

* **Configure API Access:** This section prepares your environment and API key before running experiments.

**Hints:**
- Create a Groq API key from your account
- Set `GROQ_API_KEY` in your shell before launching Jupyter
- Install dependencies in the next code cell

**What each step does:**
| Step | Description |
|------|-------------|
| `Create key` | Enables authenticated API calls |
| `export GROQ_API_KEY=...` | Stores key in environment variable for secure access |
| `pip install` | Installs required Python packages (`groq`, `pandas`) |


In [ ]:
%pip install -q groq pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 4.9 MB/s eta 0:00:00


In [ ]:
# Hint: Set your Groq key in the shell, then restart kernel if needed.
# !export GROQ_API_KEY="______"

In [ ]:
import os
import time
from typing import Dict, Any, List
from google.colab import userdata

import pandas as pd
from groq import Groq

# Hint: Prefer environment variable; direct key is only for quick testing.
api_key = userdata.get("GROQ_API_KEY")  # e.g., "GROQ_API_KEY"
if not api_key:
    raise EnvironmentError(
        "GROQ_API_KEY is not set. Set it in your shell before running this notebook."
    )

client = Groq(api_key=api_key)
print("Groq client initialized")

Groq client initialized


## 3. Configure Candidate Models

* **Model List Setup:** This cell defines which Groq models will be benchmarked on the same task.

**Hints:**
- Keep at least 3 models for comparison
- Use model IDs available in your Groq account
- Include both smaller and larger models to compare efficiency

**What each line does:**
| Line | Description |
|------|-------------|
| `MODELS = [...]` | Defines benchmark candidates |
| `assert len(MODELS) >= 3` | Enforces minimum comparison size |
| Last expression `MODELS` | Displays current model list in notebook output |

**Documentation:**
- [Groq Model IDs](https://console.groq.com/docs/models) - Available model names

In [ ]:
# Edit these as needed based on your account access.
MODELS = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "mixtral-8x7b-32768",
    # Optional examples (enable only if available in your account):
    "openai/gpt-oss-20b",
    # "openai/gpt-oss-120b",
]

assert len(MODELS) >= 3, "Use at least 3 models."
MODELS

['llama-3.1-8b-instant',
 'llama-3.3-70b-versatile',
 'mixtral-8x7b-32768',
 'openai/gpt-oss-20b']

## 4. Define Deterministic Evaluation Rule

* **Exact-Match Task:** This section defines a deterministic math prompt and strict correctness check.

**Hints:**
- Deterministic tasks reduce ambiguity in evaluation
- Exact string match makes scoring automatic and reproducible
- Normalize whitespace before comparison

**What each function does:**
| Function | Description |
|----------|-------------|
| `normalize_text(text)` | Removes extra whitespace for consistent comparison |
| `check_correctness(text, expected_answer)` | Marks response correct only when normalized text matches expected answer exactly |

**Documentation:**
- [str.split](https://docs.python.org/3/library/stdtypes.html#str.split) - Whitespace tokenization
- [str.join](https://docs.python.org/3/library/stdtypes.html#str.join) - Rejoin normalized text

In [ ]:
# Deterministic benchmark tasks (exact-match evaluation)
TASKS = [
    {
        "task_id": "math_exact_match",
        "prompt": (
            # Hint: Ask for ONLY final value to keep scoring deterministic.
            "Compute (37 * 19) - 11. "
            "Return ONLY the final integer with no words or punctuation."
        ),
        "expected_answer": "692",
    },
]

def normalize_text(text: str) -> str:
    return " ".join((text or "").strip().split())

def check_correctness(text: str, expected_answer: str) -> Dict[str, Any]:
    pred = normalize_text(text)
    gold = normalize_text(expected_answer)
    is_correct = pred == gold

    return {
        "is_correct": is_correct,
        "expected_answer": gold,
        "normalized_response": pred,
        "exact_match": is_correct,
        "word_count": len(pred.split()) if pred else 0,
    }

In [ ]:
def run_single_model(model: str, prompt: str, temperature: float = 0.0) -> Dict[str, Any]:
    t0 = time.perf_counter()
    completion = client.chat.completions.create(
        model=model,  # Hint: pass `model` argument
        temperature=temperature,  # Hint: use function argument
        messages=[
            {
                "role": "user",  # Hint: user role for prompt
                "content": prompt,  # Hint: pass `prompt`
            }
        ],
    )
    t1 = time.perf_counter()

    usage = completion.usage
    text = completion.choices[0].message.content if completion.choices else ""

    return {
        "response": text or "",
        "prompt_tokens": int(getattr(usage, "prompt_tokens", 0) or 0),
        "completion_tokens": int(getattr(usage, "completion_tokens", 0) or 0),
        "total_tokens": int(getattr(usage, "total_tokens", 0) or 0),
        "latency_sec": round(t1 - t0, 4),
    }

## 5. Single-Model Walkthrough

* **Run One Example First:** This step demonstrates the full pipeline using only one model before scaling to all models.

**Hints:**
- Use `TASKS[0]` and `MODELS[0]` for a quick sanity check
- Print expected answer and model answer
- Confirm exact-match correctness before running full loop

**What each step does:**
| Step | Description |
|------|-------------|
| Pick task/model | Selects one deterministic benchmark and one model |
| API call | Gets response and token usage metadata |
| Correctness check | Applies exact-match scoring |
| Print summary | Shows correctness, tokens, and latency |

**Documentation:**
- [f-strings](https://docs.python.org/3/tutorial/inputoutput.html#formatted-string-literals) - Structured output printing

In [ ]:
# Pick first task and first model
example_task = TASKS[0]  # Hint: first task index
example_model = MODELS[0]  # Hint: first model index

# Call model once
example_out = run_single_model(
    model=example_model,
    prompt=example_task['prompt'],  # Hint: task field containing prompt text
    temperature=0.0,  # Hint: deterministic value
 )

# Check exact-match correctness
example_check = check_correctness(
    text=example_out["response"],  # Hint: model output text field
    expected_answer=example_task['expected_answer'],  # Hint: expected answer field
)

print(f"Model: {example_model}")
print(f"Expected answer: {example_task['expected_answer']}")
print(f"Model answer: {example_out['response']}")
print(f"Correct (exact match): {example_check['is_correct']}")
print(f"Total tokens: {example_out['total_tokens']}")
print(f"Latency (sec): {example_out['latency_sec']}")

Model: llama-3.1-8b-instant
Expected answer: 692
Model answer: 37 * 19 = 703
703 - 11 = 692
Correct (exact match): False
Total tokens: 73
Latency (sec): 0.2134


In [ ]:
rows = []

for task in TASKS:
    for model in MODELS:
        record = {
            "task_id": task["task_id"],
            "model": model,
            "status": "ok",
            "response": "",
            "expected_answer": task["expected_answer"],
            "normalized_response": "",
            "exact_match": False,
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
            "latency_sec": None,
            "is_correct": False,
            "word_count": None,
            "error": "",
        }

        try:
            # Hint: call helper with current model and task prompt.
            out = run_single_model(model=model, prompt=task["prompt"], temperature=0.0)
            check = check_correctness(
                text=out["response"],
                expected_answer=task["expected_answer"],
            )

            record.update(out)  # Hint: merge output dict
            record.update(check)  # Hint: merge correctness dict
        except Exception as exc:
            record["status"] = "error"
            record["error"] = str(exc)

        rows.append(record)  # Hint: add record to list

results_df = pd.DataFrame(rows)
results_df

,task_id,model,status,response,expected_answer,normalized_response,exact_match,prompt_tokens,completion_tokens,total_tokens,latency_sec,is_correct,word_count,error
0,math_exact_match,llama-3.1-8b-instant,ok,37 * 19 = 703\n703 - 11 = 692,692,37 * 19 = 703 703 - 11 = 692,False,57.0,16.0,73.0,0.1356,False,10.0,
1,math_exact_match,llama-3.3-70b-versatile,ok,703,692,703,False,57.0,2.0,59.0,0.0915,False,1.0,
2,math_exact_match,mixtral-8x7b-32768,error,,692,,False,NaN,NaN,NaN,NaN,False,NaN,Error code: 400 - {'error': {'message': 'The m...
3,math_exact_match,openai/gpt-oss-20b,ok,692,692,692,True,93.0,63.0,156.0,0.1180,True,1.0,


In [ ]:
summary_cols = [
    "task_id",
    "model",
    "status",
    "is_correct",
    "exact_match",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "latency_sec",
    "word_count",
]

print("All Model Results")
display(results_df[summary_cols].sort_values(["task_id", "total_tokens"], na_position="last"))

print("\nDeterministic evaluation details")
preview = results_df[["task_id", "model", "expected_answer", "normalized_response", "response", "error"]].copy()
preview["response"] = preview["response"].fillna("").str.slice(0, 220)
display(preview)

All Model Results


,task_id,model,status,is_correct,exact_match,prompt_tokens,completion_tokens,total_tokens,latency_sec,word_count
1,math_exact_match,llama-3.3-70b-versatile,ok,False,False,57.0,2.0,59.0,0.0915,1.0
0,math_exact_match,llama-3.1-8b-instant,ok,False,False,57.0,16.0,73.0,0.1356,10.0
3,math_exact_match,openai/gpt-oss-20b,ok,True,True,93.0,63.0,156.0,0.1180,1.0
2,math_exact_match,mixtral-8x7b-32768,error,False,False,NaN,NaN,NaN,NaN,NaN



Deterministic evaluation details


,task_id,model,expected_answer,normalized_response,response,error
0,math_exact_match,llama-3.1-8b-instant,692,37 * 19 = 703 703 - 11 = 692,37 * 19 = 703\n703 - 11 = 692,
1,math_exact_match,llama-3.3-70b-versatile,692,703,703,
2,math_exact_match,mixtral-8x7b-32768,692,,,Error code: 400 - {'error': {'message': 'The m...
3,math_exact_match,openai/gpt-oss-20b,692,692,692,


In [ ]:
# Select best model: exact-match correct first, then minimum total tokens, then lower latency
valid = results_df[(results_df["status"] == "ok") & (results_df["is_correct"] == True)].copy()

if valid.empty:
    print("No exact-match correct outputs found.")
    best_per_task = pd.DataFrame()
else:
    best_per_task = (
        valid.sort_values(["task_id", "total_tokens", "latency_sec"], ascending=[True, True, True])
        .groupby("task_id", as_index=False)
        .first()
    )
    print("Most Efficient Correct Model Per Task")
    display(
        best_per_task[
            [
                "task_id",
                "model",
                "expected_answer",
                "normalized_response",
                "total_tokens",
                "prompt_tokens",
                "completion_tokens",
                "latency_sec",
            ]
        ]
    )

Most Efficient Correct Model Per Task


,task_id,model,expected_answer,normalized_response,total_tokens,prompt_tokens,completion_tokens,latency_sec
0,math_exact_match,openai/gpt-oss-20b,692,692,156.0,93.0,63.0,0.118


## 6. Experimental Analysis

* **Interpret Results:** Use the generated tables to analyze model efficiency and correctness.

**Hints:**
- Compare only correct models when selecting best efficiency
- Observe whether lower token usage also preserves correctness
- Compare latency differences across models

**What to analyze:**
| Question | Purpose |
|----------|---------|
| Fewest tokens? | Identify cheapest correct model |
| Quality maintained? | Validate correctness is not sacrificed |
| Latency trend? | Understand speed vs model size trade-offs |
| Prompt impact? | Evaluate if wording changes token usage |

**Documentation:**
- [Pandas sorting](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) - Rank models by metrics
- [Pandas groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) - Best-per-task selection